# 市場廣度預測 0050 報酬 v6

Breadth Dynamics & Extreme Breadth Study。核心邏輯位於 `src/market_breadth/`；本 notebook 只負責環境、執行、驗證與輸出。

In [ ]:
# Colab: clone private GitHub repo without printing the token
import base64, os, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/hh4832/taiwan-market-breadth-research.git'
BRANCH = 'main'
REPO_DIR = Path('/content/taiwan-market-breadth-research')

if Path('/content').exists():
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('請在 Colab Secrets 設定 GITHUB_TOKEN')
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    env = os.environ.copy()
    basic_auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {basic_auth}',
    })
    try:
        clone = subprocess.run(
            ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
            check=False, env=env, capture_output=True, text=True,
        )
        if clone.returncode != 0:
            safe_error = clone.stderr.replace(token, '[REDACTED]').replace(basic_auth, '[REDACTED]')
            raise RuntimeError(
                'GitHub clone 失敗。請確認 GITHUB_TOKEN 可讀取 private repo，且此 notebook 已允許存取該 Secret。\n'
                f'git stderr: {safe_error.strip()}'
            )
    finally:
        del token, basic_auth
    os.chdir(REPO_DIR)
else:
    REPO_DIR = Path.cwd()
print('Repository:', REPO_DIR)
print('Commit:', subprocess.run(['git','rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# Install the repository's declared dependencies
%pip install -q -r requirements.txt

In [ ]:
# Add src and run token-independent validation first
import sys, subprocess
sys.path.insert(0, str(REPO_DIR / 'src'))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True, env={**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src')})

In [ ]:
# FinLab login follows the installed FinLab package's normal authenticated flow
from finlab import data
probe = data.get('price:收盤價')
print('FinLab latest date:', probe.index.max(), 'shape:', probe.shape)

In [ ]:
from market_breadth.config import V6Config
from market_breadth.pipeline import run

config = V6Config()
RESULTS = run(config)
display(RESULTS['results'].head())
display(RESULTS['metadata'])

In [ ]:
# Immutable Google Drive archive
if Path('/content').exists():
    from google.colab import drive
    from market_breadth.export import archive_to_drive
    drive.mount('/content/drive')
    archive = archive_to_drive(RESULTS['output_paths'], 'taiwan-market-breadth-research')
    print('Drive archive:', archive)